# ESTA ES LA VERSIÓN DE DIAGNÓSTICO VERBOSO

**Versión:** `v0.8-websearch-debug`  
**Última modificación:** `2026-09-11 20:05 CEST`  
**Rama:** `fix/chapter2-news-agent`  
**Modelo:** `gpt-4.1-mini`

Esta versión añade una prueba directa de Responses API, otra del Agents SDK y un flujo Evaluator-Optimizer con `raw_responses` visible.


## 1. Instalar dependencias

In [ ]:
!pip install -U openai openai-agents -q

## 2. Imports y entorno

In [ ]:
import json, os, sys, importlib.metadata as im
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Literal
from urllib.parse import urlsplit
from google.colab import userdata
from openai import OpenAI
from agents import Agent, Runner, WebSearchTool, ModelSettings, ItemHelpers

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("Python:", sys.version)
print("openai:", im.version("openai"))
print("openai-agents:", im.version("openai-agents"))
print("API key:", bool(os.environ.get("OPENAI_API_KEY")))

def dump(obj):
    return obj.model_dump() if hasattr(obj, "model_dump") else obj

def urls_in(obj):
    if hasattr(obj, "model_dump"): obj=obj.model_dump()
    out=[]
    if isinstance(obj, dict):
        for k,v in obj.items():
            if k=="url" and isinstance(v,str) and v.startswith("http"): out.append(v)
            out += urls_in(v)
    elif isinstance(obj,(list,tuple)):
        for v in obj: out += urls_in(v)
    return list(dict.fromkeys(out))

def show(label, obj):
    print("\n"+"="*25, label, "="*25)
    d=dump(obj)
    print(json.dumps(d, indent=2, default=str)[:30000])
    us=urls_in(obj)
    print("\nURLs:", len(us))
    for u in us[:50]: print(" ",u)
    return us


## 3. PRUEBA A — Responses API directa (sin Agents SDK)

In [ ]:
client=OpenAI()
direct = client.responses.create(
    model="gpt-4.1-mini",
    tools=[{"type":"web_search","external_web_access":True}],
    tool_choice="required",
    include=["web_search_call.action.sources"],
    input="Search the live web. Find ONE Reuters article published on September 10 or 11, 2026 about AI, OpenAI, Nvidia, Oracle, Adobe, AI infrastructure or AI investment. Cite it."
)
print("OUTPUT TEXT:\n", direct.output_text)
print("OUTPUT TYPES:", [getattr(x,"type",None) for x in direct.output])
show("DIRECT RESPONSE RAW", direct)


## 4. PRUEBA B — Agents SDK + WebSearchTool

In [ ]:
probe=Agent(
    name="probe",
    model="gpt-4.1-mini",
    instructions="You MUST use web search. Find one Reuters article from September 10 or 11, 2026 about AI. Cite it. Do not answer from memory.",
    tools=[WebSearchTool(search_context_size="high", external_web_access=True)],
    model_settings=ModelSettings(tool_choice="required",response_include=["web_search_call.action.sources"])
)
pr=await Runner.run(probe,"Search now.")
print("FINAL OUTPUT:\n",pr.final_output)
print("NEW ITEM CLASSES:",[type(x).__name__ for x in pr.new_items])
print("RAW ITEM TYPES:",[getattr(getattr(x,"raw_item",None),"type",None) for x in pr.new_items])
print("RAW RESPONSES:",len(pr.raw_responses))
for i,r in enumerate(pr.raw_responses):
    show(f"AGENTS RAW RESPONSE {i}",r)


## 5. Flujo Evaluator-Optimizer con diagnóstico

In [ ]:
today=datetime.now().strftime("%Y-%m-%d")
start=(datetime.now()-timedelta(days=2)).strftime("%Y-%m-%d")

searcher=Agent(
    name="web_news_searcher",
    model="gpt-4.1-mini",
    instructions=f'''You MUST use web search and never answer from memory.
Find genuine Reuters articles between {start} and {today}.
For each item provide headline, date, summary and inline citation.
Search requested companies/topics separately if needed. Do not fabricate.''',
    tools=[WebSearchTool(search_context_size="high",external_web_access=True)],
    model_settings=ModelSettings(tool_choice="required",response_include=["web_search_call.action.sources"])
)

@dataclass
class Eval:
    feedback:str
    score:Literal["successful","unsuccessful"]

evaluator=Agent(
    name="evaluator",
    model="gpt-4.1-mini",
    instructions=f'''Strictly evaluate the original request. Successful only if the requested number of Reuters articles is present, each with headline/date/summary, dates between {start} and {today}, and genuine Reuters URL evidence. Zero results is always unsuccessful.''',
    output_type=Eval
)

async def main():
    q=input("User's request: " ).strip()
    feedback=None
    last=""
    for n in range(1,3):
        prompt=q if feedback is None else q+"\nPrevious failure: "+feedback+"\nRun a NEW web search."
        r=await Runner.run(searcher,prompt)
        last=ItemHelpers.text_message_outputs(r.new_items)
        print(f"\n***** NEWS SEARCH {n} *****\n{last}")
        print("new_items:",[(type(x).__name__,getattr(getattr(x,"raw_item",None),"type",None)) for x in r.new_items])
        allurls=[]
        for rr in r.raw_responses:
            allurls += urls_in(rr)
        allurls=list(dict.fromkeys(allurls))
        reuters=[u for u in allurls if "reuters.com" in urlsplit(u).netloc.lower()]
        print("RAW RESPONSES:",len(r.raw_responses))
        print("ALL URLS:",len(allurls))
        for u in allurls[:50]: print(" SOURCE:",u)
        print("REUTERS URLS:",len(reuters))
        for u in reuters: print(" REUTERS:",u)
        for i,rr in enumerate(r.raw_responses):
            print(f"\n--- RAW RESPONSE {i} (first 30000 chars) ---")
            print(json.dumps(dump(rr),indent=2,default=str)[:30000])
        ev=await Runner.run(evaluator,f"ORIGINAL REQUEST:\n{q}\n\nANSWER:\n{last}\n\nREUTERS URLS:\n"+("\n".join(reuters) if reuters else "NONE"))
        res:Eval=ev.final_output
        print("EVALUATOR:",res.score,res.feedback)
        if res.score=="successful": break
        feedback=res.feedback
    print("\nFINAL:\n",last)


## 6. Ejecutar

In [ ]:
await main()